# RSNA Knee — Modeling v20: k-fold CV utk laterality canonicalization (konfirmasi v19)

**Konteks**: v19 (single-split, cache preprocessing v4/laterality canon) = 0,8245 -- sedikit di
bawah v15 (0,8304, cache v2) tapi jauh di atas v18 OOF (0,8068, cache v2 5-fold). Selisih dari v15
(-0,0059) PRAKTIS TIDAK BISA DIBEDAKAN dari noise antar-fold yg terukur di v18 (std 0,034, rentang
0,078) -- v19 inconclusive di single-split (`PROGRESS.md` §32).

**v20 ini = v18 (struktur k-fold, 5 fold, OOF di 57 gold) TAPI pakai cache preprocessing v4**
(laterality canonicalization) -- SATU-SATUNYA beda dari v18 adalah `kernel_sources` data. Metrik
yg dibandingkan: **OOF v20 vs OOF v18 (0,8068)** -- ini perbandingan apples-to-apples yg valid
(sama2 metrik OOF 5-fold, bukan single-split vs OOF spt v19 vs v15).

**Kalau OOF v20 > 0,8068 secara meyakinkan** (bukan cuma dalam noise antar-fold): laterality
canonicalization TERBUKTI pengungkit nyata -- checkpoint v20 jadi model final baru proyek,
gantikan v15/v18. **Kalau OOF v20 <= 0,8068 atau bedanya kecil**: canonicalization tidak terbukti
membantu di arsitektur mean-pooling kita -- kembali ke v15/v18 sbg model final, catat sbg
pelajaran (mean-pooling tidak eksploitasi laterality spt attention-pooling sumber referensi).

**Arsitektur/hyperparameter SAMA PERSIS dgn v18**: DINOv2-base, unfreeze 6, LR head/backbone
terpisah, `CosineAnnealingLR` per-fold (direset tiap fold), EPOCHS=8, K_SLICE=4, TPU, 5-fold CV
(gold dibagi 5, (K-1)/K gold masuk training tiap fold + 1/K jadi validasi OOF, tiap fold dari nol).

**Estimasi waktu**: ~4,5 jam (sama spt v18).


## 0. Setup

In [ ]:
import os, sys, json, time, random, traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

T0 = time.time()
def tlog(msg):
    print(f"[t={time.time()-T0:7.1f}s] {msg}", flush=True)

def ensure(pkg, import_name=None):
    import importlib
    name = import_name or pkg
    try:
        importlib.import_module(name); return True
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q {pkg}")
        try:
            importlib.import_module(name); return True
        except Exception as e:
            print(f"[warn] gagal import/install {pkg}: {e}"); return False

ensure("transformers")
ensure("scikit-learn", "sklearn")  # v13: image TPU belum pernah dipakai proyek ini, jangan asumsikan preinstalled
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# v13: deteksi torch_xla (TPU) SEBELUM cek CUDA -- kuota GPU 0 sampai 22 Agustus, TPU yg jadi
# target utama run ini. Fallback CUDA lalu CPU tetap dipertahankan (pola sama dgn _cuda_usable()
# lama) biar notebook ini tetap aman dijalankan ulang di GPU nanti tanpa perlu diedit balik.
XLA_OK = False
xm = None
try:
    import torch_xla.core.xla_model as xm
    _xla_dev = xm.xla_device()
    _ = torch.zeros(1, device=_xla_dev) + 1  # sanity: device benar2 usable, bukan cuma modul ke-import
    XLA_OK = True
except Exception as e:
    print(f"[info] torch_xla tidak tersedia/tidak usable ({e}) -- fallback cek CUDA/CPU.")

def _cuda_usable():
    """metadata minta machine_shape=NvidiaTeslaT4, tapi kalau suatu saat Kaggle tetap kasih GPU
    lama (mis. P100, compute capability sm_60) yang tidak didukung build PyTorch terpasang, jangan
    crash -- fallback CPU dgn peringatan jelas, drpd gagal total di tengah training."""
    if not torch.cuda.is_available():
        return False
    try:
        _ = torch.zeros(1, device="cuda") + 1
        return True
    except Exception as e:
        print(f"[warn] GPU terdeteksi tapi tidak kompatibel dgn build PyTorch ini ({e}) -- fallback CPU.")
        return False

if XLA_OK:
    DEVICE = _xla_dev
elif _cuda_usable():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
tlog(f"device: {DEVICE} | xla: {XLA_OK} | gpu_count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")

OUT_DIR = Path("model_outputs")
OUT_DIR.mkdir(exist_ok=True)


## 1. Cari sumber data (preprocessing cache, weak labels, DINOv2 weights)

In [ ]:
kaggle_input = Path("/kaggle/input")

def find_dir_containing(filename, hint=""):
    """Cari folder di /kaggle/input yang mengandung file tertentu via os.walk dgn pruning subtree
    besar (image_cache, train_series, test_series) -- kernel_sources ternyata mount lebih dalam dari
    dugaan awal (mis. /kaggle/input/notebooks/<owner>/<slug>/...), jadi glob dangkal tidak cukup.
    Pola sama persis dgn find_dinov2() yg sudah terbukti aman/cepat di dataset besar ini."""
    if not kaggle_input.exists():
        return None
    candidates = []
    for root, dirs, files in os.walk(kaggle_input):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series", "image_cache")]
        if filename in files:
            candidates.append(Path(root))
    if not candidates:
        return None
    if hint:
        for c in candidates:
            if hint.lower() in str(c).lower():
                return c
    return candidates[0]

PREP_DIR = find_dir_containing("series_manifest.csv", "preprocessing")
LABELER_DIR = find_dir_containing("weak_labels_full.csv", "labeler")
LLM_DIR = find_dir_containing("report_labels_v4hybrid.csv", "hybrid-report-labels")
tlog(f"PREP_DIR: {PREP_DIR}")
tlog(f"LABELER_DIR: {LABELER_DIR}")
tlog(f"LLM_DIR: {LLM_DIR}")

def find_dinov2(variant="base"):
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series", "image_cache")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None

DINOV2_DIR = find_dinov2("base")
tlog(f"DINOV2_DIR: {DINOV2_DIR}")

# v15: checkpoint lama (v11/v14) TIDAK di-resume -- fokus eksperimen ini murni mengisolasi
# variabel geometri data (preprocessing v2 vs v1), bukan soal resume. CKPT_DIR akan None
# (tidak apa2, sudah ditangani fallback "training dari nol" di bawah) krn kernel_sources
# sengaja tidak menunjuk kernel modeling manapun kali ini.
CKPT_DIR = find_dir_containing("best_model.pt", "modeling")
tlog(f"CKPT_DIR: {CKPT_DIR}")

assert PREP_DIR is not None, "preprocessing kernel belum di-attach (kernel_sources)"
assert LABELER_DIR is not None, "weak-labeler kernel belum di-attach (kernel_sources)"
assert LLM_DIR is not None, "dataset public hybrid-report-labels belum di-attach (dataset_sources)"
assert DINOV2_DIR is not None, "DINOv2 model belum di-attach (model_sources)"


## 2. Load & gabungkan: label, manifest series, cache path

**Update v6 -- sumber label diganti.** Riset tambahan (lihat `docs/RESEARCH_NOTES.md` §7) mengukur
kualitas beberapa sumber weak-label report-text terhadap 58 studi gold kita:

| sumber | macro AUC vs gold |
|---|---|
| rule-based labeler kita sendiri (estimasi dari precision/recall/agreement) | ~0.71 |
| `stevenleehans/rsna-knee-llm-report-labels` v1 (llm_labels_full.csv) | 0.8780 (terverifikasi ulang, cocok persis dgn klaim publikasi) |
| `stevenleehans/rsna-knee-llm-report-labels` v4_blend | 0.8927 (terverifikasi ulang, cocok persis) |
| `flight0234/rsna-knee-hybrid-report-labels` (v4_blend + kolom Fracture dari pilkwang v2) | **0.8991** (dipakai mulai v8, terverifikasi ulang independen) |

v4_blend adalah dataset publik CC0-1.0 (public domain), berisi label **probabilitas lunak [0,1]**
per finding hasil pembacaan LLM atas kolom `Report` (tidak ada data di luar kompetisi ini yg dipakai
oleh pembuatnya) -- jauh lebih akurat drpd labeler rule-based kita sendiri, terutama utk label yg
selama ini lemah (Synovitis, Contusion, Effusion). **Dipakai sbg target training utk 4.349 studi
non-gold**; 57 studi gold TETAP pakai label asli (hard 0/1) dari annotator, tidak pernah ditimpa oleh
estimasi apa pun. Barun2104/rsna-knee-stratified-folds-and-llm-soft-labels di-skip -- setelah dicek,
kolom labelnya identik persis (selisih absolut = 0.0) dgn 58 studi gold kita sendiri utk baris
berlabel, dan NaN utk sisanya -- bukan sumber informasi baru, cuma re-package train.csv resmi.

In [ ]:
series_manifest = pd.read_csv(PREP_DIR / "series_manifest.csv")
study_manifest = pd.read_csv(PREP_DIR / "study_manifest.csv")
weak_labels = pd.read_csv(LABELER_DIR / "weak_labels_full.csv")
llm_labels = pd.read_csv(LLM_DIR / "report_labels_v4hybrid.csv")

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

tlog(f"series_manifest: {series_manifest.shape} | study_manifest: {study_manifest.shape} "
     f"| weak_labels: {weak_labels.shape} | llm_labels: {llm_labels.shape}")

# Exclude 1 studi gold-label yang ter-flag anomali (OVEREXPOSED sistemik) dari audit exhaustive kita
EXCLUDE_STUDIES = {"1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921"}
weak_labels = weak_labels[~weak_labels["StudyInstanceUID"].isin(EXCLUDE_STUDIES)].reset_index(drop=True)
series_manifest = series_manifest[series_manifest["status"] == "OK"].reset_index(drop=True)

# Gabung: basis = label LLM (soft [0,1], utk semua studi) -> lalu 57 studi gold ditimpa dgn label asli.
gold_mask = weak_labels["source"] == "gold"
CONF_COLS = [c + "__conf" for c in LABEL_COLS]
train_labels = llm_labels[["StudyInstanceUID"] + LABEL_COLS + CONF_COLS].merge(
    weak_labels[["StudyInstanceUID", "lang"]], on="StudyInstanceUID", how="right")
train_labels = train_labels.set_index("StudyInstanceUID")
gold_rows = weak_labels.loc[gold_mask, ["StudyInstanceUID"] + LABEL_COLS].set_index("StudyInstanceUID")
train_labels.loc[gold_rows.index, LABEL_COLS] = gold_rows[LABEL_COLS]
train_labels["source"] = np.where(train_labels.index.isin(gold_rows.index), "gold", "flight0234_v4hybrid")
# studi gold pakai label asli (hard 0/1) -> confidence penuh (1.0), bukan nilai __conf dataset publik
for c in CONF_COLS:
    train_labels.loc[gold_rows.index, c] = 1.0
train_labels = train_labels.reset_index()

tlog(f"Setelah exclude anomali & gabung label: {train_labels.shape[0]} studi label")
tlog(f"Studi gold bersih: {(train_labels['source']=='gold').sum()}")


## 3. Skema slot (plane x sekuens) + prior anatomis per-finding

In [ ]:
# 6 slot: {Sagittal, Coronal, Axial} x {struktural (non fluid-sensitive), fluid-sensitive+fat-sat}
def slot_index(plane, fs, fatsup):
    plane = str(plane)
    is_fluid = (int(fs) == 1) if pd.notna(fs) else False
    base = {"Sagittal": 0, "Coronal": 1, "Axial": 2}.get(plane, 0)
    return base + (3 if is_fluid else 0)

N_SLOT = 6
SLOT_NAMES = ["Sagittal-struct", "Coronal-struct", "Axial-struct", "Sagittal-fluid", "Coronal-fluid", "Axial-fluid"]

# Prior domain-knowledge kita sendiri (lihat docs/RESEARCH_NOTES.md) -- soft bias, bukan hard constraint.
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (0, 1), "Lateral OA": (0, 1), "PF OA": (2, 5),
    "Effusion": (3, 4, 5), "Synovitis": (3, 4, 5),
    "Baker\'s": (2, 3, 5), "Contusion": (3, 4, 5), "Fracture": (0, 1, 2),
}
SLOT_PRIOR_STRENGTH = 0.55

series_manifest["slot"] = series_manifest.apply(
    lambda r: slot_index(r["Anatomical_Plane"], r["Fluid_Sensitive"], r["Fat_Suppression"]), axis=1)
tlog("Distribusi slot:")
print(series_manifest["slot"].value_counts().sort_index().rename(lambda i: SLOT_NAMES[i]))


## 4. PyTorch Dataset: 1 studi -> bag 6 slot (rata-rata antar-series & antar-slice dalam slot)

In [ ]:
K_SLICE = 4  # subsample dari 16 slice cache/series -- cukup utk v1, kompute lebih murah
IMG_SIZE = 224

# index: studi -> {slot: [cache_path, ...]}  (bisa >1 series per slot -> nanti dirata-rata)
#
# BUG v4/v5 (ketemu via diagnostik 7b -- 100% mask kosong & 100% gambar kosong): kolom "cache_path" di
# series_manifest.csv menyimpan path RELATIF apa adanya dari saat kernel preprocessing jalan
# (mis. "prep_outputs/image_cache/<study>/<series>.npz", relatif ke /kaggle/working kernel ITU),
# bukan path absolut. Saat notebook ini pakai ulang lewat kernel_sources, cwd berbeda & prefix mount
# juga beda (/kaggle/input/notebooks/<owner>/<slug>/...) -- np.load(cache_path) makanya selalu
# FileNotFoundError, ketangkep diam2 oleh except Exception: continue di _load_slot(). Fix: rekonstruksi
# path dari PREP_DIR (yg auto-discovery-nya SUDAH benar per-kernel) + struktur relatif yg diketahui,
# jangan percaya string cache_path yang tersimpan.
study_to_series = defaultdict(lambda: defaultdict(list))
n_missing_file = 0
for _, r in series_manifest.iterrows():
    if r.get("status") != "OK":
        continue
    p = PREP_DIR / "image_cache" / str(r["StudyInstanceUID"]) / f'{r["SeriesInstanceUID"]}.npz'
    if p.exists():
        study_to_series[r["StudyInstanceUID"]][r["slot"]].append(p)
    else:
        n_missing_file += 1
tlog(f"study_to_series: {len(study_to_series)} studi terindeks | {n_missing_file} file cache tidak ketemu (dari {len(series_manifest)} series OK)")

class KneeStudyDataset(Dataset):
    def __init__(self, study_ids, labels_df):
        self.study_ids = list(study_ids)
        self.labels = labels_df.set_index("StudyInstanceUID")

    def __len__(self):
        return len(self.study_ids)

    def _load_slot(self, paths):
        """Rata-rata (mean pixel) antar-series dalam slot yang sama, subsample K slice."""
        slices = []
        for p in paths[:2]:  # maks 2 series/slot dirata-rata, cukup utk v1
            try:
                arr = np.load(p)["slices"]  # (16, 224, 224) uint8
                idx = np.linspace(0, arr.shape[0] - 1, min(K_SLICE, arr.shape[0])).astype(int)
                slices.append(arr[idx])
            except Exception:
                continue
        if not slices:
            return None
        stacked = np.stack(slices, axis=0).astype(np.float32).mean(axis=0)  # (K_SLICE,224,224)
        return stacked

    def __getitem__(self, idx):
        study = self.study_ids[idx]
        imgs = np.zeros((N_SLOT, K_SLICE, IMG_SIZE, IMG_SIZE), dtype=np.float32)
        mask = np.zeros(N_SLOT, dtype=np.float32)
        for slot, paths in study_to_series.get(study, {}).items():
            arr = self._load_slot(paths)
            if arr is not None:
                imgs[slot] = arr
                mask[slot] = 1.0

        row = self.labels.loc[study] if study in self.labels.index else None
        target = np.full(len(LABEL_COLS), np.nan, dtype=np.float32)
        weight = np.zeros(len(LABEL_COLS), dtype=np.float32)
        if row is not None:
            for i, c in enumerate(LABEL_COLS):
                v = row.get(c)
                if pd.notna(v):
                    target[i] = float(v)
                    # v8: pakai kolom __conf eksplisit dari flight0234 (confidence asli dari LLM
                    # reader, bukan heuristik 2*|p-0.5| yg cuma proxy). Utk gold, __conf sudah
                    # di-set 1.0 di sel gabungan label (§2).
                    cv = row.get(c + "__conf")
                    weight[i] = float(cv) if pd.notna(cv) else 1.0

        return {"study": study,
                "imgs": torch.from_numpy(imgs),      # (N_SLOT, K_SLICE, H, W) float32 [0,255]
                "mask": torch.from_numpy(mask),       # (N_SLOT,)
                "target": torch.from_numpy(target),   # (12,) float32, NaN = tidak berlabel
                "weight": torch.from_numpy(weight)}   # (12,) float32, confidence utk BCE


## 5. Arsitektur model

In [ ]:
LABEL_KEY = [c.replace("\'", "") for c in LABEL_COLS]  # kunci aman utk dict (Baker\'s -> Bakers)
SLOT_PRIOR_IDX = {k.replace("\'", ""): v for k, v in SLOT_PRIOR_TABLE.items()}

class SlotHead(nn.Module):
    """Attention per-diagnosis atas embedding 6 slot, dgn prior anatomis lembut."""
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        prior = torch.zeros(n_out, n_slot)
        for i, target in enumerate(LABEL_KEY):
            for s in SLOT_PRIOR_IDX.get(target, ()):
                prior[i, s] = SLOT_PRIOR_STRENGTH
        self.register_buffer("slot_prior", prior)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb                                # (B, S, H)
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class KneeModel(nn.Module):
    def __init__(self, backbone_path, unfreeze_last=2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(str(backbone_path))
        # v11: OOM di v10 (unfreeze_last=6, BATCH_STUDIES=8, T4 14.5GB) -- 6 blok trainable
        # butuh jauh lebih banyak memori aktivasi drpd 2 blok. Gradient checkpointing menukar
        # compute (recompute aktivasi saat backward) demi memori -- aman utk unfreeze lebih dalam.
        # v13a run pertama ERROR: AttributeError: module 'torch' has no attribute 'xla' --
        # torch.utils.checkpoint._get_device_module() coba getattr(torch, 'xla') pas forward pass
        # (bukan pas enable() dipanggil, makanya try/except di sini TIDAK cukup), krn versi
        # torch/transformers ini belum expose submodule torch.xla utk device-module lookup HF grad
        # checkpointing. Fix: matikan grad checkpointing KHUSUS di device XLA, tetap aktif di
        # CUDA/CPU spt v11/v12 (device global ini terisi sebelum KneeModel dibuat, lihat §0).
        if DEVICE.type == 'xla':
            print("[info] grad checkpointing DINONAKTIFKAN di device XLA -- torch.utils.checkpoint "
                  "belum support device 'xla' di versi ini (AttributeError, lihat PROGRESS.md v13).")
        else:
            try:
                self.backbone.gradient_checkpointing_enable()
            except Exception as e:
                print(f"[warn] gradient_checkpointing_enable() gagal ({e}) -- lanjut TANPA grad checkpointing.")
        n_layer = len(self.backbone.encoder.layer)
        for prm in self.backbone.parameters():
            prm.requires_grad = False
        for blk in self.backbone.encoder.layer[max(0, n_layer - unfreeze_last):]:
            for prm in blk.parameters():
                prm.requires_grad = True
        for prm in self.backbone.layernorm.parameters():
            prm.requires_grad = True
        dim = self.backbone.config.hidden_size
        self.head = SlotHead(dim * 2, N_SLOT, len(LABEL_COLS))   # x2: CLS token + mean-patch dikonkat di forward()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"backbone: {n_layer} blok, {unfreeze_last} terakhir trainable, dim={dim}, "
              f"total trainable params={n_trainable/1e6:.1f}M")

    def forward(self, imgs, mask):
        B, S, K, H, W = imgs.shape
        x = imgs.reshape(B * S * K, 1, H, W).div(255.0)
        x = x.repeat(1, 3, 1, 1)                    # grayscale cache -> 3 channel utk backbone ImageNet-style
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state   # (B*S*K, tokens, dim)
        cls = out[:, 0]
        patch_mean = out[:, 1:].mean(1)
        feat = torch.cat([cls, patch_mean], dim=1)               # (B*S*K, 2*dim)
        feat = feat.reshape(B, S, K, -1).mean(2)                 # rata-rata antar-slice dalam slot -> (B, S, 2*dim)
        return self.head(feat, mask)


## 6. Sanity check arsitektur (tensor dummy, sebelum load data asli)

In [ ]:
# Cek forward pass jalan & shape benar SEBELUM training beneran -- murah, tangkap bug lebih awal.
_dummy_imgs = torch.randint(0, 256, (2, N_SLOT, K_SLICE, IMG_SIZE, IMG_SIZE), dtype=torch.float32)
_dummy_mask = torch.ones(2, N_SLOT); _dummy_mask[1, -1] = 0.0
try:
    _m = KneeModel(DINOV2_DIR, unfreeze_last=6).to(DEVICE)
    with torch.no_grad():
        _out = _m(_dummy_imgs.to(DEVICE), _dummy_mask.to(DEVICE))
    assert _out.shape == (2, len(LABEL_COLS)), f"shape salah: {_out.shape}"
    tlog(f"Sanity check OK. Output shape: {tuple(_out.shape)}")
    del _m, _out
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
except Exception as e:
    tlog(f"SANITY CHECK GAGAL: {e}")
    raise


## 7. Split train/val (stratifikasi di level studi, gold dipisah utk validasi bersih)

In [ ]:
# v18: k-fold CV -- gold TIDAK lagi 100% dibuang ke validasi. GOLD_STUDIES dibagi K_FOLDS grup;
# per fold, (K-1)/K grup gold masuk TRAINING (ekstra sinyal label asli) + 1/K grup jadi validasi
# OOF. Dataset per-fold dibangun di dalam training loop (§8), bukan di sini -- di sini cuma
# tentukan keanggotaan studi & fold assignment-nya.
GOLD_STUDIES = train_labels.loc[train_labels["source"] == "gold", "StudyInstanceUID"].tolist()
WEAK_STUDIES = train_labels.loc[train_labels["source"] == "flight0234_v4hybrid", "StudyInstanceUID"].tolist()

# hanya studi yang punya >=1 series ter-cache
GOLD_STUDIES = [s for s in GOLD_STUDIES if s in study_to_series]
WEAK_STUDIES = [s for s in WEAK_STUDIES if s in study_to_series]
tlog(f"Studi gold (dibagi k-fold): {len(GOLD_STUDIES)} | studi label-LLM (training semua fold): {len(WEAK_STUDIES)}")
assert len(WEAK_STUDIES) > 0, "WEAK_STUDIES kosong -- cek string filter 'source' cocok dgn nilai di §2"
assert len(GOLD_STUDIES) > 0, "GOLD_STUDIES kosong -- cek label gold ter-gabung dgn benar di §2"

K_FOLDS = 5
_fold_rng = np.random.RandomState(20260819)  # seed tetap -> fold assignment reproducible
_gold_shuffled = list(GOLD_STUDIES)
_fold_rng.shuffle(_gold_shuffled)
FOLD_VAL_IDS = [_gold_shuffled[i::K_FOLDS] for i in range(K_FOLDS)]  # interleaved -> ukuran merata
for _i, _f in enumerate(FOLD_VAL_IDS):
    tlog(f"  fold {_i}: {len(_f)} studi gold validasi")
assert sum(len(f) for f in FOLD_VAL_IDS) == len(GOLD_STUDIES), "fold assignment tidak menutupi semua studi gold"


## 7b. Diagnostik (murah, sebelum training)

**Riwayat:** run v4 (full, 5 epoch) mentok val_auc=0.50 persis di semua epoch walau train_loss turun.
Diagnostik v5 (`RUN_FULL_TRAINING=False`) menemukan akar masalahnya: **100% studi (train & val)
punya mask kosong (0 slot terisi) dan gambar kosong (mean pixel ~0)** -- model memang tidak pernah
melihat gambar sama sekali, cuma belajar bias per-label dari label (makanya loss BCE turun dikit
walau AUC tetap chance level). Root cause: kolom `cache_path` di `series_manifest.csv` menyimpan path
relatif dari kernel preprocessing asli, tidak valid lagi saat dipakai ulang lewat `kernel_sources` di
notebook lain -- np.load() gagal diam2 (exception ketangkep broad `except`). **Sudah diperbaiki** di
sel index `study_to_series` (§4) dgn merekonstruksi path dari `PREP_DIR` yang auto-discovery-nya benar
per-kernel. Sel di bawah ini menjalankan diagnostik yg sama sekali lagi utk KONFIRMASI fix-nya benar2
jalan SEBELUM commit ke training penuh (berjam-jam, kuota GPU) -- kalau mask/gambar masih kosong,
training di bawah TIDAK akan dijalankan biar tidak buang kuota lagi.

In [ ]:
# 1) Statistik mask & label -- murni CPU, cepat.
def dataset_stats(ds, name, n_sample=200):
    n = min(n_sample, len(ds))
    idxs = np.random.RandomState(0).choice(len(ds), n, replace=False)
    mask_sums, target_valid_counts, img_means = [], [], []
    for i in idxs:
        item = ds[i]
        mask_sums.append(item["mask"].sum().item())
        target_valid_counts.append((~torch.isnan(item["target"])).sum().item())
        img_means.append(item["imgs"].mean().item())
    mask_sums = np.array(mask_sums); tvc = np.array(target_valid_counts); img_means = np.array(img_means)
    print(f"--- {name} (n={n}) ---")
    print(f"  mask.sum() per studi: mean={mask_sums.mean():.2f} min={mask_sums.min():.0f} max={mask_sums.max():.0f} "
          f"| studi dgn mask KOSONG (0 slot terisi): {(mask_sums==0).sum()} ({(mask_sums==0).mean()*100:.1f}%)")
    print(f"  label valid (non-NaN) per studi: mean={tvc.mean():.2f} | studi TANPA label sama sekali: {(tvc==0).sum()} ({(tvc==0).mean()*100:.1f}%)")
    print(f"  rata2 pixel image (harus > 0 kalau gambar ke-load beneran): mean={img_means.mean():.2f} "
          f"| studi dgn gambar KOSONG (mean~0): {(img_means < 1.0).sum()} ({(img_means<1.0).mean()*100:.1f}%)")
    return float((img_means < 1.0).mean())

# v18: train_ds/val_ds global tidak ada lagi (dataset sekarang dibangun per-fold di §8) --
# diagnostik ini cuma sanity check sistem-wide (cache kebaca, gambar tidak kosong), pakai
# WEAK_STUDIES & GOLD_STUDIES langsung, bukan fold tertentu.
_diag_train_ds = KneeStudyDataset(WEAK_STUDIES, train_labels)
_diag_val_ds = KneeStudyDataset(GOLD_STUDIES, train_labels)
_train_blank_frac = dataset_stats(_diag_train_ds, "train_ds (semua weak-label)")
_val_blank_frac = dataset_stats(_diag_val_ds, "val_ds (semua gold)")
DATA_OK = (_train_blank_frac < 0.5) and (_val_blank_frac < 0.5)
tlog(f"DATA_OK = {DATA_OK} (fraksi gambar kosong: train={_train_blank_frac*100:.1f}%, val={_val_blank_frac*100:.1f}%)")


In [ ]:
# 2) Variasi embedding antar-studi -- forward pass kecil (~16 studi), TANPA training.
_diag_model = KneeModel(DINOV2_DIR, unfreeze_last=6).to(DEVICE)
_diag_model.eval()
_diag_loader = DataLoader(_diag_train_ds, batch_size=8, shuffle=True, num_workers=2)  # v18: pakai _diag_train_ds (§7b)
_feats_list, _preds_list, _targets_list = [], [], []
with torch.no_grad():
    for bi, batch in enumerate(_diag_loader):
        if bi >= 2:
            break
        imgs, mask, target = batch["imgs"].to(DEVICE), batch["mask"].to(DEVICE), batch["target"].to(DEVICE)
        B, S, K, H, W = imgs.shape
        x = imgs.reshape(B * S * K, 1, H, W).div(255.0).repeat(1, 3, 1, 1)
        x = (x - _diag_model.mean) / _diag_model.std
        out = _diag_model.backbone(pixel_values=x).last_hidden_state
        cls = out[:, 0]; patch_mean = out[:, 1:].mean(1)
        feat = torch.cat([cls, patch_mean], dim=1).reshape(B, S, K, -1).mean(2)   # (B,S,2*dim)
        logits = _diag_model.head(feat, mask)
        _feats_list.append(feat.mean(1).cpu())      # rata-ratakan slot -> (B, 2*dim) per studi
        _preds_list.append(torch.sigmoid(logits).cpu())
        _targets_list.append(target.cpu())

FEATS = torch.cat(_feats_list)          # (N, 2*dim)
PREDS = torch.cat(_preds_list).numpy()  # (N, 12)
TARGETS_DIAG = torch.cat(_targets_list).numpy()

# Jarak antar-embedding studi -- kalau semua studi mirip/identik, ini masalah nyata (data bug).
pairwise_std = FEATS.std(dim=0).mean().item()
between_study_var = FEATS.var(dim=0).mean().item()
print(f"Variasi embedding antar-studi (rata2 std per dimensi): {pairwise_std:.4f}")
print(f"  -> kalau ini mendekati 0, backbone menghasilkan fitur nyaris SAMA utk semua studi (bug data-loading / normalisasi berlebih)")
print(f"\nRentang prediksi model (setelah sigmoid) per label:")
print(pd.DataFrame(PREDS, columns=LABEL_COLS).describe().loc[["mean","std","min","max"]].T)
print(f"\n-> kalau kolom \'std\' semua ~0: model memang cuma keluarkan angka konstan per label (bias-only, tdk pakai gambar sama sekali)")

del _diag_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 8. Loss (masked + confidence-weighted BCE) & training loop

In [ ]:
def masked_bce(logits, target, weight=None):
    mask = ~torch.isnan(target)
    if mask.sum() == 0:
        return logits.sum() * 0.0
    t = torch.where(mask, target, torch.zeros_like(target))
    loss = F.binary_cross_entropy_with_logits(logits, t, reduction="none")
    w = mask.float()
    if weight is not None:
        w = w * torch.where(mask, weight, torch.zeros_like(weight))
    return (loss * w).sum() / w.sum().clamp_min(1e-6)

def macro_auc(y_true, y_pred):
    from sklearn.metrics import roc_auc_score
    scores = []
    for j in range(y_true.shape[1]):
        col_true, col_pred = y_true[:, j], y_pred[:, j]
        valid = ~np.isnan(col_true)
        if valid.sum() < 2 or len(np.unique(col_true[valid])) < 2:
            continue
        scores.append(roc_auc_score(col_true[valid], col_pred[valid]))
    return float(np.mean(scores)) if scores else float("nan"), scores

# v18: k-fold CV -- SATU-SATUNYA perbedaan struktural dari v15 (semua hyperparameter per-epoch
# identik). Tiap fold dilatih dari NOL (bobot pretrained murni, TIDAK saling resume) supaya
# independen & OOF benar2 valid ("belum pernah lihat studi validasinya").
EPOCHS = 8
BATCH_STUDIES = 4
LR_HEAD = 1e-3
LR_BACKBONE = 8e-6
SAFETY_CUTOFF_SECONDS = 7.0 * 3600  # k-fold: ~5x lebih lama dari 1 fold (v15 ~55 menit) -> ~4.2 jam + margin
RUN_FULL_TRAINING = True

fold_results = []
oof_pred, oof_true = {}, {}   # StudyInstanceUID -> np.array(12,) -- prediksi/label OOF per studi gold

if not RUN_FULL_TRAINING:
    tlog("RUN_FULL_TRAINING=False -- lewati training loop, ini cuma push diagnostik.")
elif not DATA_OK:
    tlog("DATA_OK=False -- gambar/mask masih kosong massal, training DIBATALKAN biar tidak buang kuota. "
         "Cek ulang path cache_path/PREP_DIR sebelum push berikutnya.")
else:
    for fold_idx in range(K_FOLDS):
        if time.time() - T0 > SAFETY_CUTOFF_SECONDS:
            tlog(f"BATAS WAKTU AMAN TERCAPAI -- berhenti sebelum fold {fold_idx} mulai "
                 f"({fold_idx}/{K_FOLDS} fold selesai).")
            break
        tlog(f"===== FOLD {fold_idx+1}/{K_FOLDS} =====")

        val_ids_f = FOLD_VAL_IDS[fold_idx]
        val_id_set_f = set(val_ids_f)
        # (K-1)/K studi gold (semua fold LAIN) ditambahkan ke training -- ekstra sinyal label asli,
        # bukan cuma dibuang ke validasi spt skema v15 lama.
        extra_gold_train_f = [s for s in GOLD_STUDIES if s not in val_id_set_f]
        train_ids_f = WEAK_STUDIES + extra_gold_train_f
        train_ds_f = KneeStudyDataset(train_ids_f, train_labels)
        val_ds_f = KneeStudyDataset(val_ids_f, train_labels)
        tlog(f"  train_ds: {len(train_ds_f)} ({len(WEAK_STUDIES)} weak + {len(extra_gold_train_f)} gold) "
             f"| val_ds (OOF): {len(val_ds_f)}")

        model = KneeModel(DINOV2_DIR, unfreeze_last=6).to(DEVICE)
        backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        opt = torch.optim.AdamW([
            {"params": head_params, "lr": LR_HEAD},
            {"params": backbone_params, "lr": LR_BACKBONE},
        ], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS))
        train_loader = DataLoader(train_ds_f, batch_size=BATCH_STUDIES, shuffle=True, num_workers=2, drop_last=True)
        val_loader = DataLoader(val_ds_f, batch_size=BATCH_STUDIES, shuffle=False, num_workers=2)

        best_auc_f = -1.0
        best_state_f, best_val_true_f, best_val_pred_f = None, None, None
        history_f = []
        for epoch in range(EPOCHS):
            if time.time() - T0 > SAFETY_CUTOFF_SECONDS:
                tlog(f"  BATAS WAKTU AMAN TERCAPAI di tengah fold {fold_idx} -- berhenti sebelum epoch berikutnya.")
                break
            model.train()
            running = 0.0
            for bi, batch in enumerate(train_loader):
                imgs, mask, target = batch["imgs"].to(DEVICE), batch["mask"].to(DEVICE), batch["target"].to(DEVICE)
                weight = batch["weight"].to(DEVICE)
                logits = model(imgs, mask)
                loss = masked_bce(logits, target, weight)
                opt.zero_grad(); loss.backward(); opt.step()
                if XLA_OK:
                    xm.mark_step()
                running += loss.item()
                if (bi + 1) % 100 == 0:
                    tlog(f"  fold {fold_idx+1} epoch {epoch+1} batch {bi+1}/{len(train_loader)} loss={running/(bi+1):.4f}")

            model.eval()
            all_true, all_pred = [], []
            with torch.no_grad():
                for batch in val_loader:
                    imgs, mask, target = batch["imgs"].to(DEVICE), batch["mask"].to(DEVICE), batch["target"].to(DEVICE)
                    logits = model(imgs, mask)
                    all_true.append(target.cpu().numpy())
                    all_pred.append(torch.sigmoid(logits).cpu().numpy())
            y_true = np.concatenate(all_true) if all_true else np.zeros((0, len(LABEL_COLS)))
            y_pred = np.concatenate(all_pred) if all_pred else np.zeros((0, len(LABEL_COLS)))
            auc, per_label = macro_auc(y_true, y_pred)
            cur_lr_head = opt.param_groups[0]["lr"]
            cur_lr_backbone = opt.param_groups[1]["lr"]
            history_f.append({"fold": fold_idx, "epoch": epoch + 1, "train_loss": running / max(1, len(train_loader)),
                               "val_macro_auc": auc, "lr_head": cur_lr_head, "lr_backbone": cur_lr_backbone})
            tlog(f"  FOLD {fold_idx+1} EPOCH {epoch+1}/{EPOCHS} selesai. train_loss={running/max(1,len(train_loader)):.4f} "
                 f"val_macro_auc={auc:.4f} lr_head={cur_lr_head:.2e} lr_backbone={cur_lr_backbone:.2e}")

            if auc == auc and auc > best_auc_f:  # auc==auc -> bukan NaN
                best_auc_f = auc
                best_state_f = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_val_true_f, best_val_pred_f = y_true, y_pred
                tlog(f"    -> checkpoint fold {fold_idx+1} disimpan (val_macro_auc={auc:.4f})")

            scheduler.step()

        # simpan checkpoint TERBAIK fold ini + catat prediksi OOF (dari epoch terbaik fold ini)
        if best_state_f is not None:
            torch.save(best_state_f, OUT_DIR / f"best_model_fold{fold_idx}.pt")
            for sid, pred_row, true_row in zip(val_ds_f.study_ids, best_val_pred_f, best_val_true_f):
                oof_pred[sid] = pred_row
                oof_true[sid] = true_row

        fold_results.append({"fold": fold_idx, "best_val_macro_auc": best_auc_f, "n_val": len(val_ds_f)})
        pd.DataFrame(history_f).to_csv(OUT_DIR / f"training_history_fold{fold_idx}.csv", index=False)
        tlog(f"FOLD {fold_idx+1}/{K_FOLDS} SELESAI. Best val macro AUC: {best_auc_f:.4f}")

        del model, opt, scheduler
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    # --- Ringkasan lintas-fold + OOF gabungan ---
    pd.DataFrame(fold_results).to_csv(OUT_DIR / "fold_results.csv", index=False)
    fold_aucs = [r["best_val_macro_auc"] for r in fold_results if r["best_val_macro_auc"] == r["best_val_macro_auc"]]
    if fold_aucs:
        tlog(f"Rata-rata AUC per-fold: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f} "
             f"(min={min(fold_aucs):.4f}, max={max(fold_aucs):.4f}, n_fold={len(fold_aucs)})")

    if oof_pred:
        oof_ids = list(oof_pred.keys())
        oof_y_true = np.stack([oof_true[s] for s in oof_ids])
        oof_y_pred = np.stack([oof_pred[s] for s in oof_ids])
        oof_auc, oof_per_label = macro_auc(oof_y_true, oof_y_pred)
        tlog(f"OOF MACRO AUC (gabungan {len(oof_ids)}/{len(GOLD_STUDIES)} studi gold, tiap studi divalidasi "
             f"model yg TIDAK PERNAH melihatnya): {oof_auc:.4f}")
        oof_df = pd.DataFrame({"StudyInstanceUID": oof_ids})
        for i, c in enumerate(LABEL_COLS):
            oof_df[f"pred_{c}"] = oof_y_pred[:, i]
            oof_df[f"true_{c}"] = oof_y_true[:, i]
        oof_df.to_csv(OUT_DIR / "oof_predictions.csv", index=False)
        tlog(f"Perbandingan: v15 (1 split N=57) = 0.8304 | v18 OOF (5-fold, N={len(oof_ids)}) = {oof_auc:.4f}")
    else:
        tlog("[warn] tidak ada prediksi OOF tersimpan (kemungkinan SAFETY_CUTOFF_SECONDS tercapai sebelum fold pertama selesai).")


## Catatan v18

- **k-fold CV proper (K=5)** atas 57 studi gold -- setiap studi gold divalidasi tepat sekali oleh
  model yg tidak pernah melihatnya (OOF), sambil tetap dapat manfaat sbg training di 4 fold lain.
- **Metrik utama sekarang: OOF macro AUC** (gabungan 5 fold, cakup semua 57 gold) -- lebih stabil
  drpd 1 split N=57 (v15 dst.). Rata-rata + std AUC per-fold juga dicatat (indikasi variansi/noise).
- 5 checkpoint tersimpan (best_model_fold0.pt .. best_model_fold4.pt) -- bisa jadi ensemble 5-model
  kalau OOF-nya bagus & anggotanya cukup beragam.
- Tiap fold dilatih DARI NOL (bobot pretrained murni), TIDAK saling resume -- independensi fold
  penting utk validitas OOF.
- Kalau OOF macro AUC mendekati/melampaui 0,8304 (v15): validasi v15 sebelumnya representatif,
  bukan kebetulan split yg menguntungkan. Kalau jauh beda: v15 kemungkinan sebagian besar
  keberuntungan split, model "sebenarnya" ada di sekitar angka OOF ini.
